In [1]:
################################################
# Jegadeesh & Titman (1993) Momentum Portfolio #
# June 2019                                     #  
# Qingyi (Freda) Song Drechsler                #
# August 2021                                  #
# Alex Malek                                   #
################################################

import pandas as pd
import numpy as np
import wrds
import matplotlib.pyplot as plt
from pandas.tseries.offsets import *
from scipy import stats

###################
# Connect to WRDS #
###################
conn=wrds.Connection(user_name="zkg232")

###################
# CRSP Block      #
###################
# sql similar to crspmerge macro
# added exchcd=-2,-1,0 to address the issue that stocks temp stopped trading
# without exchcd=-2,-1, 0 the non-trading months will be tossed out in the output
# leading to wrong cumret calculation in momentum step
# Code	Definition
# -2	Halted by the NYSE or AMEX
# -1	Suspended by the NYSE, AMEX, or NASDAQ
# 0	Not Trading on NYSE, AMEX, or NASDAQ
# 1	New York Stock Exchange
# 2	American Stock Exchange

crsp_m = conn.raw_sql("""
                      select a.permno, a.permco, b.ncusip, a.date, 
                      b.shrcd, b.exchcd, b.siccd,
                      a.ret, a.vol, a.shrout, a.prc, a.cfacpr, a.cfacshr
                      from crsp.msf as a
                      left join crsp.msenames as b
                      on a.permno=b.permno
                      and b.namedt<=a.date
                      and a.date<=b.nameendt
                      where a.date between '01/01/1963' and '12/31/2025'
                      and b.exchcd between -2 and 2
                      and b.shrcd between 10 and 11
                      """) 

# Change variable format to int
crsp_m[['permco','permno','shrcd','exchcd']]=\
    crsp_m[['permco','permno','shrcd','exchcd']].astype(int)

# Line up date to be end of month
crsp_m['date']=pd.to_datetime(crsp_m['date'])


WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [2]:
#######################################################
# Create Momentum Portfolio                           #   
# Measures Based on Past (J) Month Compounded Returns #
#######################################################

J = 6 # Formation Period Length: J can be between 3 to 12 months
K = 6 # Holding Period Length: K can be between 3 to 12 months

_tmp_crsp = crsp_m[['permno','date','ret']].sort_values(['permno','date'])\
    .set_index('date')

# Replace missing return with 0
_tmp_crsp['ret']=_tmp_crsp['ret'].fillna(0)

# Calculate rolling cumulative return
# by summing log(1+ret) over the formation period
_tmp_crsp['logret']=np.log(1+_tmp_crsp['ret'])
umd = _tmp_crsp.groupby(['permno'])['logret'].rolling(J, min_periods=J).sum()
umd = umd.reset_index()
umd['cumret']=np.exp(umd['logret'])-1


In [3]:
########################################
# Formation of 10 Momentum Portfolios  #
########################################

# For each date: assign ranking 1-10 based on cumret
# 1=lowest 10=highest cumret
umd=umd.dropna(axis=0, subset=['cumret'])
umd['momr']=umd.groupby('date')['cumret'].transform(lambda x: pd.qcut(x, 10, labels=False))

umd.momr=umd.momr.astype(int)
umd['momr'] = umd['momr']+1

# Corrected previous version month end line up issue
# First lineup date to month end date medate
# Then calculate hdate1 and hdate2 using medate

umd['form_date'] = umd['date']
umd['medate'] = umd['date']+MonthEnd(0)
umd['hdate1']=umd['medate']+MonthBegin(1)
umd['hdate2']=umd['medate']+MonthEnd(K)
umd = umd[['permno', 'form_date','momr','hdate1','hdate2']]

In [4]:
tmp1 = umd
tmp1['hdate1_m'] = pd.DatetimeIndex(tmp1['hdate1']).month
tmp1['hdate2_m'] = pd.DatetimeIndex(tmp1['hdate2']).month

In [5]:
tmp1.head()

,permno,form_date,momr,hdate1,hdate2,hdate1_m,hdate2_m
5,10001,2010-05-28,9,2010-06-01,2010-11-30,6,11
6,10001,2010-06-30,8,2010-07-01,2010-12-31,7,12
7,10001,2010-07-30,8,2010-08-01,2011-01-31,8,1
8,10001,2010-08-31,8,2010-09-01,2011-02-28,9,2
9,10001,2010-09-30,8,2010-10-01,2011-03-31,10,3


In [6]:
tmp1['mgap'] = tmp1.hdate2_m - tmp1.hdate1_m
tmp1['mgap'] = np.where(tmp1.mgap>0, tmp1.mgap, tmp1.mgap+12)
tmp1.head(10)

,permno,form_date,momr,hdate1,hdate2,hdate1_m,hdate2_m,mgap
5,10001,2010-05-28,9,2010-06-01,2010-11-30,6,11,5
6,10001,2010-06-30,8,2010-07-01,2010-12-31,7,12,5
7,10001,2010-07-30,8,2010-08-01,2011-01-31,8,1,5
8,10001,2010-08-31,8,2010-09-01,2011-02-28,9,2,5
9,10001,2010-09-30,8,2010-10-01,2011-03-31,10,3,5
10,10001,2010-10-29,7,2010-11-01,2011-04-30,11,4,5
11,10001,2010-11-30,2,2010-12-01,2011-05-31,12,5,5
12,10001,2010-12-31,2,2011-01-01,2011-06-30,1,6,5
13,10001,2011-01-31,2,2011-02-01,2011-07-31,2,7,5
14,10001,2011-02-28,3,2011-03-01,2011-08-31,3,8,5


In [7]:
%%time
# join rank and return data together
# note: the merge step consumes a lot of memory
# split the larger dataframe into chunks size, and then filter concat, repeat
_tmp_ret = crsp_m[['permno','date','ret']]

# These 2 operations replaced w/ loop below
#port = pd.merge(_tmp_ret, umd, on=['permno'], how='inner')
#port = port[(port['hdate1']<=port['date']) & (port['date']<=port['hdate2'])]
port = pd.DataFrame()
n = 100000
for i in range(0, _tmp_ret.shape[0], n): 
    chunks =   _tmp_ret.iloc[i:i + n]
    merged = umd.merge(chunks, on=['permno'], how='inner')
    merged = merged[(merged['hdate1']<=merged['date']) & (merged['date']<=merged['hdate2'])]
    port = pd.concat([port, merged] )

umd2 = port.sort_values(by=['date','momr','form_date','permno']).drop_duplicates()
umd3 = umd2.groupby(['date','momr','form_date'])['ret'].mean().reset_index()

CPU times: user 18.7 s, sys: 12.2 s, total: 30.8 s
Wall time: 32.5 s


In [8]:
# Skip first two years of the sample 
start_yr = umd3['date'].dt.year.min()+2
umd3 = umd3[umd3['date'].dt.year>=start_yr]
umd3 = umd3.sort_values(by=['date','momr'])

# Create one return series per MOM group every month
ewret = umd3.groupby(['date','momr'])['ret'].mean().reset_index()
ewstd = umd3.groupby(['date','momr'])['ret'].std().reset_index()
ewret = ewret.rename(columns={'ret':'ewret'})
ewstd = ewstd.rename(columns={'ret':'ewretstd'})
ewretdat = pd.merge(ewret, ewstd, on=['date','momr'], how='inner')
ewretdat = ewretdat.sort_values(by=['momr'])

# portfolio summary
ewretdat.groupby(['momr'])['ewret'].describe()[['count','mean', 'std']]


,count,mean,std
momr,,,
1,720.0,0.009854,0.098466
2,720.0,0.010456,0.0701
3,720.0,0.011474,0.061672
4,720.0,0.011696,0.056054
5,720.0,0.012218,0.052936
6,720.0,0.012136,0.050926
7,720.0,0.012284,0.050109
8,720.0,0.01238,0.051683
9,720.0,0.013281,0.054757


In [9]:
#################################
# Long-Short Portfolio Returns  #
#################################

# Transpose portfolio layout to have columns as portfolio returns
ewretdat2 = ewretdat.pivot(index='date', columns='momr', values='ewret')

# Add prefix port in front of each column
ewretdat2 = ewretdat2.add_prefix('port')
ewretdat2 = ewretdat2.rename(columns={'port1':'losers', 'port10':'winners'})
ewretdat2['long_short'] = ewretdat2['winners'] - ewretdat2['losers']

# Compute Long-Short Portfolio Cumulative Returns
ewretdat3 = ewretdat2
ewretdat3['1+losers']=1+ewretdat3['losers']
ewretdat3['1+winners']=1+ewretdat3['winners']
ewretdat3['1+ls'] = 1+ewretdat3['long_short']

ewretdat3['cumret_winners']=ewretdat3['1+winners'].cumprod()-1
ewretdat3['cumret_losers']=ewretdat3['1+losers'].cumprod()-1
ewretdat3['cumret_long_short']=ewretdat3['1+ls'].cumprod()-1

#################################
# Portfolio Summary Statistics  #
################################# 

# Mean 
mom_mean = ewretdat3[['winners', 'losers', 'long_short']].mean().to_frame()
mom_mean = mom_mean.rename(columns={0:'mean'}).reset_index()

# T-Value and P-Value
t_losers = pd.Series(stats.ttest_1samp(ewretdat3['losers'],0.0)).to_frame().T
t_winners = pd.Series(stats.ttest_1samp(ewretdat3['winners'],0.0)).to_frame().T
t_long_short = pd.Series(stats.ttest_1samp(ewretdat3['long_short'],0.0)).to_frame().T

t_losers['momr']='losers'
t_winners['momr']='winners'
t_long_short['momr']='long_short'

t_output =pd.concat([t_winners, t_losers, t_long_short])\
    .rename(columns={0:'t-stat', 1:'p-value'})

# Combine mean, t and p
mom_output = pd.merge(mom_mean, t_output, on=['momr'], how='inner')

In [10]:
ewretdat3

momr,losers,port2,port3,port4,port5,port6,port7,port8,port9,winners,long_short,1+losers,1+winners,1+ls,cumret_winners,cumret_losers,cumret_long_short
date,,,,,,,,,,,,,,,,,
1965-01-29,0.125506,0.087303,0.07182,0.066853,0.063503,0.059245,0.057853,0.057473,0.060759,0.075909,-0.049597,1.125506,1.075909,0.950403,0.075909,0.125506,-0.049597
1965-02-26,0.023589,0.036167,0.039597,0.02983,0.031593,0.029587,0.031388,0.035125,0.04355,0.032706,0.009117,1.023589,1.032706,1.009117,0.111097,0.152055,-0.040932
1965-03-31,0.029698,0.013216,0.012294,0.016013,0.011194,0.004046,0.007166,0.011911,0.018315,0.023156,-0.006542,1.029698,1.023156,0.993458,0.136826,0.186269,-0.047206
1965-04-30,0.030817,0.043124,0.041973,0.033889,0.037783,0.041774,0.040272,0.044803,0.052073,0.05339,0.022573,1.030817,1.05339,1.022573,0.197521,0.222827,-0.025699
1965-05-28,-0.018265,-0.01268,-0.009051,-0.00872,-0.007953,-0.006805,-0.004244,-0.00093,0.005152,0.004448,0.022713,0.981735,1.004448,1.022713,0.202847,0.200491,-0.00357
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-30,-0.089749,-0.037228,-0.01457,-0.006483,0.005573,0.003505,0.003809,0.002044,0.005013,0.002673,0.092423,0.910251,1.002673,1.092423,9471.07963,36.21415,6.248846
2024-09-30,0.004169,0.006089,0.006544,0.012215,0.010196,0.010863,0.011588,0.018342,0.014234,0.022051,0.017882,1.004169,1.022051,1.017882,9679.94657,36.369285,6.37847
2024-10-31,-0.021857,-0.000631,-0.027858,-0.013613,-0.015863,-0.008721,-0.012036,-0.010102,-0.001463,0.004364,0.026221,0.978143,1.004364,1.026221,9722.196324,35.552516,6.57194


In [11]:
mom_output

,momr,mean,t-stat,p-value
0,winners,0.015168,6.224327,8.215064e-10
1,losers,0.009854,2.685237,7.414938e-03
2,long_short,0.005314,2.114699,3.479915e-02
